# LLM Reproducibility Evaluation Framework
## Step 1: Dataset Loading & Preprocessing
本notebook实现三个轻量级benchmark（SST-2, AG News, IMDb）的加载、清洗和prompt格式化。
这是整个评估框架的基础，后续的zero-shot/few-shot实验都依赖这里的输出。

In [ ]:
# ============================================================
# 依赖库导入
# datasets: HuggingFace数据集库，用于加载标准benchmark
# re: 正则表达式，用于清洗IMDb中的HTML标签
# collections.Counter: 用于majority baseline计算
# ============================================================
from datasets import load_dataset
from collections import Counter
import re

In [ ]:
# ============================================================
# 加载三个数据集
# SST-2: 斯坦福情感分析（正面/负面），来自电影评论
# AG News: 新闻主题分类（4类：World/Sports/Business/Sci-Tech）
# IMDb: 电影评论情感分析（正面/负面），文本较长
# ============================================================
sst2 = load_dataset("stanfordnlp/sst2")
ag   = load_dataset("ag_news")
imdb = load_dataset("imdb")

print("数据集加载完成")
print(f"SST-2 validation size : {len(sst2['validation'])}")
print(f"AG News test size     : {len(ag['test'])}")
print(f"IMDb test size        : {len(imdb['test'])}")

数据集加载完成
SST-2 validation size : 872
AG News test size     : 7600
IMDb test size        : 25000


In [ ]:
# ============================================================
# 全局配置：标签映射 & Prompt模板
#
# LABEL_MAPPINGS: 将数据集中的数字标签转为可读字符串
#   - 用于构造few-shot exemplar和解析模型输出
#
# PROMPT_TEMPLATES: zero-shot prompt模板
#   - {input} 占位符会被实际文本替换
#   - 模板措辞固定，保证跨实验可比性（reproducibility核心）
#   - 注意：模板设计本身也是一个实验变量，后续会系统测试
# ============================================================
LABEL_MAPPINGS = {
    "sst2"   : {0: "negative", 1: "positive"},
    "imdb"   : {0: "negative", 1: "positive"},
    "ag_news": {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}
}

PROMPT_TEMPLATES = {
    "sst2"   : "Classify the sentiment of the following review:\n{input}\nLabel:",
    "imdb"   : "Classify the sentiment of the following movie review:\n{input}\nLabel:",
    "ag_news": "Classify the topic of the following news article into World, Sports, Business, or Sci/Tech:\n{input}\nLabel:"
}

In [ ]:
# ============================================================
# 预处理函数
#
# 输入: HuggingFace dataset的单条example + 数据集名称
# 输出: 包含以下字段的dict
#   - raw_text         : 清洗后的原始文本
#   - formatted_prompt : 填入模板后的完整prompt
#   - true_label_id    : 数字标签（用于计算accuracy）
#   - true_label_name  : 文字标签（用于few-shot exemplar构造）
#
# 注意事项:
#   1. IMDb含有HTML标签（<br />等），需要清洗，否则干扰模型理解
#   2. 截断在单词边界而非字符边界，避免截断到单词中间
#   3. max_chars=500 对SST2/AG够用；IMDb较长，视需要可调整
# ============================================================
def preprocess_data(example, dataset_name, max_chars=500):

    # 统一读取文本字段（SST-2用'sentence'，其他用'text'）
    raw_text = example.get("text", example.get("sentence", ""))

    # 清洗HTML标签（主要针对IMDb，其他数据集无影响）
    raw_text = re.sub(r'<.*?>', ' ', raw_text)

    # 去除首尾空白
    raw_text = raw_text.strip()

    # 长度截断：在单词边界截断，避免截到词中间
    if len(raw_text) > max_chars:
        raw_text = raw_text[:max_chars].rsplit(' ', 1)[0]

    # 将文本填入对应数据集的prompt模板
    prompt = PROMPT_TEMPLATES[dataset_name].format(input=raw_text)

    # 获取标签（数字→文字）
    label_id   = int(example["label"])
    label_name = LABEL_MAPPINGS[dataset_name][label_id]

    return {
        "raw_text"        : raw_text,
        "formatted_prompt": prompt,
        "true_label_id"   : label_id,
        "true_label_name" : label_name
    }

In [ ]:
# ============================================================
# 对测试集应用预处理
#
# 重要：实验评估应使用test/validation集，而非train集
#   - SST-2 → validation（官方test集标签未公开）
#   - AG News → test
#   - IMDb → test
# ============================================================
sst2_test_pp = sst2["validation"].map(lambda x: preprocess_data(x, "sst2"))
ag_test_pp   = ag["test"].map(lambda x: preprocess_data(x, "ag_news"))
imdb_test_pp = imdb["test"].map(lambda x: preprocess_data(x, "imdb"))

# 快速检验输出格式是否正确
print("=== SST-2 sample ===")
print("Prompt  :", sst2_test_pp[0]["formatted_prompt"])
print("Label   :", sst2_test_pp[0]["true_label_name"])

print("\n=== AG News sample ===")
print("Prompt  :", ag_test_pp[0]["formatted_prompt"])
print("Label   :", ag_test_pp[0]["true_label_name"])

print("\n=== IMDb sample (检查HTML是否已清洗) ===")
print("Prompt  :", imdb_test_pp[0]["formatted_prompt"][:300])
print("Label   :", imdb_test_pp[0]["true_label_name"])

=== SST-2 sample ===
Prompt  : Classify the sentiment of the following review:
it 's a charming and often affecting journey .
Label:
Label   : positive

=== AG News sample ===
Prompt  : Classify the topic of the following news article into World, Sports, Business, or Sci/Tech:
Fears for T N pension after talks Unions representing workers at Turner   Newall say they are 'disappointed' after talks with stricken parent firm Federal Mogul.
Label:
Label   : Business

=== IMDb sample (检查HTML是否已清洗) ===
Prompt  : Classify the sentiment of the following movie review:
I love sci-fi and am willing to put up with a lot. Sci-fi movies/TV are usually underfunded, under-appreciated and misunderstood. I tried to like this, I really did, but it is to good TV sci-fi as Babylon 5 is to Star Trek (the original). Silly p
Label   : negative


In [ ]:
# ============================================================
# 固定评估样本抽取
#
# 为什么需要固定样本？
#   - 保证所有模型、所有prompting策略都在完全相同的100条数据上评估
#   - 这是reproducibility的核心要求之一
#   - seed一旦确定，整个项目中不再更改
#
# n=100 的选择理由：
#   - 足够小，可以在有限计算资源下多次重复运行
#   - 足够大，统计结果有一定意义
#   - 符合proposal中resource-constrained的设定
# ============================================================
EVAL_SEED = 42   # 全项目固定，不要修改
EVAL_N    = 100  # 每个数据集的评估样本数

def get_eval_sample(dataset_pp, n=EVAL_N, seed=EVAL_SEED):
    """从预处理后的数据集中抽取固定的评估子集。"""
    return dataset_pp.shuffle(seed=seed).select(range(n))

sst2_eval = get_eval_sample(sst2_test_pp)
ag_eval   = get_eval_sample(ag_test_pp)
imdb_eval = get_eval_sample(imdb_test_pp)

print(f"SST-2 eval size : {len(sst2_eval)}")
print(f"AG News eval size: {len(ag_eval)}")
print(f"IMDb eval size  : {len(imdb_eval)}")

SST-2 eval size : 100
AG News eval size: 100
IMDb eval size  : 100


In [ ]:
# ============================================================
# Majority Baseline
#
# 作用：验证pipeline是否正常工作，同时建立最低性能基准
#   - 如果LLM的accuracy低于majority baseline，说明模型基本失效
#   - 对于均衡数据集（如SST-2），baseline约为50%
#   - AG News有4个均衡类别，baseline约为25%
# ============================================================
def majority_baseline_accuracy(dataset_pp, label_key="true_label_id"):
    """计算majority class baseline的accuracy。"""
    labels = [int(x[label_key]) for x in dataset_pp]
    majority_label = Counter(labels).most_common(1)[0][0]
    acc = sum(1 for y in labels if y == majority_label) / len(labels)
    return majority_label, round(acc, 4)

print("Majority Baseline Accuracy (100-sample eval sets):")
print(f"  SST-2  : {majority_baseline_accuracy(sst2_eval)}")
print(f"  AG News: {majority_baseline_accuracy(ag_eval)}")
print(f"  IMDb   : {majority_baseline_accuracy(imdb_eval)}")

Majority Baseline Accuracy (100-sample eval sets):
  SST-2  : (1, 0.57)
  AG News: (2, 0.32)
  IMDb   : (0, 0.53)


## 下一步

数据准备完成后，下一个notebook将实现：
1. **Zero-shot评估循环** — 调用模型API，记录predictions + latency + token usage
2. **Experiment Logger** — 将每次运行的配置和结果保存为JSON，确保可复现
3. **Few-shot exemplar构造** — 从train set中抽取固定exemplar，测试label ordering效果

## Week3 - 4


In [ ]:
# Load BART model
from transformers import pipeline
import time

print("正在加载模型，大约需要1-2分钟...")
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=-1  # CPU模式，Colab免费版可跑
)
print("模型加载完成 ✓")

# 快速验证模型正常
test_result = classifier(
    "This movie was absolutely wonderful!",
    candidate_labels=["positive", "negative"]
)
print(f"模型测试 → 预测: {test_result['labels'][0]}  置信度: {test_result['scores'][0]:.4f}")

正在加载模型，大约需要1-2分钟...


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

模型加载完成 ✓
模型测试 → 预测: positive  置信度: 0.9965


In [ ]:
def parse_label(raw_output, dataset_name):
    """
    将模型输出解析为标准label。
    返回标准label字符串，或 "PARSE_ERROR"
    """
    valid_labels = list(LABEL_MAPPINGS[dataset_name].values())
    raw_lower = raw_output.strip().lower()
    for label in valid_labels:
        if label.lower() in raw_lower:
            return label
    return "PARSE_ERROR"

# 测试
print(parse_label("positive", "sst2"))        # → positive
print(parse_label("Negative", "sst2"))        # → negative
print(parse_label("Sports", "ag_news"))       # → Sports
print(parse_label("xyz", "sst2"))             # → PARSE_ERROR

positive
negative
Sports
PARSE_ERROR


In [ ]:
def run_inference_bart(example, dataset_name):
    """对单条样本运行BART zero-shot推理"""
    candidate_labels = list(LABEL_MAPPINGS[dataset_name].values())

    t_start = time.time()
    result = classifier(
        example["raw_text"],       # BART直接用raw_text，不用formatted_prompt
        candidate_labels=candidate_labels
    )
    latency = time.time() - t_start

    top_label_raw    = result["labels"][0] # 返回第一个（最可能）的标签
    predicted_label  = parse_label(top_label_raw, dataset_name) # 再把标签转换成统一格式

    return {
        "predicted_label" : predicted_label,
        "true_label"      : example["true_label_name"],
        "correct"         : predicted_label == example["true_label_name"],
        "confidence"      : round(result["scores"][0], 4),
        "latency_sec"     : round(latency, 4),
        "tokens_used"     : -1   # BART没有token usage，占位用
    }

# 测试单条
r = run_inference_bart(sst2_eval[0], "sst2")
print(f"文本(前60字): {sst2_eval[0]['raw_text'][:100]}...")
print(f"真实标签: {r['true_label']}  | 预测标签: {r['predicted_label']}  | 正确: {r['correct']}")

文本(前60字): it gets onto the screen just about as much of the novella as one could reasonably expect , and is en...
真实标签: positive  | 预测标签: positive  | 正确: True


In [ ]:
import json, os
from datetime import datetime

os.makedirs("experiment_logs", exist_ok=True)

def save_experiment_log(model_name, dataset_name, prompting_strategy,
                        temperature, seed, results, accuracy, f1):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_id    = f"{model_name}_{dataset_name}_{prompting_strategy}_{timestamp}"
    latencies = [r["latency_sec"] for r in results]

    log = {
        # 实验配置（保证可复现）
        "run_id"             : run_id,
        "timestamp"          : timestamp,
        "model"              : model_name,
        "dataset"            : dataset_name,
        "prompting_strategy" : prompting_strategy,
        "temperature"        : temperature,
        "eval_seed"          : seed,
        "eval_n"             : len(results),
        # 性能指标
        "accuracy"           : round(accuracy, 4),
        "f1_macro"           : round(f1, 4),
        "avg_latency_sec"    : round(sum(latencies) / len(latencies), 4),
        "total_latency_sec"  : round(sum(latencies), 4),
        "parse_error_count"  : sum(1 for r in results if r["predicted_label"] == "PARSE_ERROR"),
        # 每条样本详细结果
        "per_sample_results" : results
    }

    filepath = f"experiment_logs/{run_id}.json"
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(log, f, ensure_ascii=False, indent=2)
    print(f"Log has been stored: {filepath}")
    return filepath

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

def run_evaluation(eval_dataset, dataset_name,
                   model_name="bart-large-mnli",
                   prompting_strategy="zero_shot",
                   temperature=0.0, seed=EVAL_SEED,
                   max_samples=None):

    dataset = eval_dataset
    if max_samples is not None:
        dataset = eval_dataset.select(range(max_samples))

    print(f"\n{'='*50}")
    print(f"Model: {model_name}  DateSet: {dataset_name}  Number of samples: {len(dataset)}")
    print(f"{'='*50}")

    results = []
    for i, example in enumerate(dataset):
        if i % 10 == 0: # 每10条打印进度
            print(f"  Progress: {i}/{len(dataset)}...")
        results.append(run_inference_bart(example, dataset_name)) # 对每条样本调用 run_inference_bart 进行预测

    true_labels = [r["true_label"]      for r in results]
    pred_labels = [r["predicted_label"] for r in results]

    # 过滤PARSE_ERROR再算指标
    valid = [(t, p) for t, p in zip(true_labels, pred_labels) if p != "PARSE_ERROR"]
    true_v, pred_v = zip(*valid)

    acc = accuracy_score(true_v, pred_v)
    f1  = f1_score(true_v, pred_v, average="macro")

    print(f"\n  Accuracy : {acc:.4f} ({acc*100:.1f}%)")
    print(f"  F1-macro : {f1:.4f}")
    print(f"  解析错误 : {sum(1 for p in pred_labels if p == 'PARSE_ERROR')} 条")

    save_experiment_log(model_name, dataset_name, prompting_strategy,
                        temperature, seed, results, acc, f1)
    return acc, f1, results

In [ ]:
# 先只跑10条，确认整个pipeline正常！
acc, f1, results = run_evaluation(
    eval_dataset=sst2_eval,
    dataset_name="sst2",
    max_samples=10
)

print("\n前3条详细：")
for r in results[:3]:
    print(f"  真实:{r['true_label']:10s} | 预测:{r['predicted_label']:10s} | 正确:{r['correct']} | 耗时:{r['latency_sec']}s")


Model: bart-large-mnli  DateSet: sst2  Number of samples: 10
  Progress: 0/10...

  Accuracy : 0.9000 (90.0%)
  F1-macro : 0.8901
  解析错误 : 0 条
Log has been stored: experiment_logs/bart-large-mnli_sst2_zero_shot_20260317_040943.json

前3条详细：
  真实:positive   | 预测:positive   | 正确:True | 耗时:1.7099s
  真实:positive   | 预测:positive   | 正确:True | 耗时:1.5596s
  真实:positive   | 预测:positive   | 正确:True | 耗时:1.352s


In [ ]:
# 全量
# 全量正式评估：3个数据集各100条
print("开始正式评估（全量100条）...")

acc_sst2, f1_sst2, _ = run_evaluation(
    sst2_eval, "sst2", max_samples=None
)

acc_ag, f1_ag, _ = run_evaluation(
    ag_eval, "ag_news", max_samples=None
)

acc_imdb, f1_imdb, _ = run_evaluation(
    imdb_eval, "imdb", max_samples=None
)

print("\n" + "="*50)
print("BART zero-shot baseline 汇总")
print("="*50)
print(f"{'数据集':<12} {'Accuracy':>10} {'F1-macro':>10} {'Majority':>10}")
print(f"{'─'*42}")
print(f"{'SST-2':<12} {acc_sst2:>10.4f} {f1_sst2:>10.4f} {'0.5700':>10}")
print(f"{'AG News':<12} {acc_ag:>10.4f} {f1_ag:>10.4f} {'0.3200':>10}")
print(f"{'IMDb':<12} {acc_imdb:>10.4f} {f1_imdb:>10.4f} {'0.5300':>10}")

开始正式评估（全量100条）...

Model: bart-large-mnli  DateSet: sst2  Number of samples: 100
  Progress: 0/100...
  Progress: 10/100...
  Progress: 20/100...
  Progress: 30/100...
  Progress: 40/100...
  Progress: 50/100...
  Progress: 60/100...
  Progress: 70/100...
  Progress: 80/100...
  Progress: 90/100...

  Accuracy : 0.8800 (88.0%)
  F1-macro : 0.8796
  解析错误 : 0 条
Log has been stored: experiment_logs/bart-large-mnli_sst2_zero_shot_20260317_035412.json

Model: bart-large-mnli  DateSet: ag_news  Number of samples: 100
  Progress: 0/100...
  Progress: 10/100...
  Progress: 20/100...
  Progress: 30/100...
  Progress: 40/100...
  Progress: 50/100...
  Progress: 60/100...
  Progress: 70/100...
  Progress: 80/100...
  Progress: 90/100...

  Accuracy : 0.7400 (74.0%)
  F1-macro : 0.7248
  解析错误 : 0 条
Log has been stored: experiment_logs/bart-large-mnli_ag_news_zero_shot_20260317_040030.json

Model: bart-large-mnli  DateSet: imdb  Number of samples: 100
  Progress: 0/100...
  Progress: 10/100...
  Pr

In [ ]:
!pip install transformers accelerate bitsandbytes -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.0 MB/s eta 0:00:00


In [ ]:
# 加载Qwen2.5
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

print("正在加载模型，大约需要2-3分钟...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model_qwen = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,  # 半精度，节省显存
    device_map="auto"           # 自动分配到GPU
)

print(f"模型加载完成 ✓")
print(f"使用设备: {next(model_qwen.parameters()).device}")

正在加载模型，大约需要2-3分钟...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

模型加载完成 ✓
使用设备: cuda:0


In [ ]:
# Qwen推理函数
def run_inference_qwen(example, dataset_name):
    """对单条样本运行Qwen2.5 zero-shot推理"""

    # Qwen是对话模型，需要用chat格式
    messages = [
        {
            "role": "system",
            "content": "You are a text classification assistant. Reply with only the label, nothing else."
        },
        {
            "role": "user",
            "content": example["formatted_prompt"]
        }
    ]

    # 把对话格式转成模型需要的输入
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer([text], return_tensors="pt").to(model_qwen.device)

    t_start = time.time()

    with torch.no_grad():  # 推理时不需要计算梯度，节省显存
        outputs = model_qwen.generate(
            **inputs,
            max_new_tokens=10,   # 只生成label，不需要长回复
            temperature=0.01,    # 接近0但避免greedy decode的边界问题
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    latency = time.time() - t_start

    # 只取模型新生成的部分，去掉输入的prompt
    input_length = inputs["input_ids"].shape[1]
    generated_ids = outputs[0][input_length:]
    raw_output = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    predicted_label = parse_label(raw_output, dataset_name)

    return {
        "predicted_label" : predicted_label,
        "true_label"      : example["true_label_name"],
        "correct"         : predicted_label == example["true_label_name"],
        "confidence"      : -1,
        "latency_sec"     : round(latency, 4),
        "tokens_used"     : len(generated_ids),
        "raw_output"      : raw_output
    }

# 测试单条
r = run_inference_qwen(sst2_eval[0], "sst2")
print(f"真实标签: {r['true_label']}  | 预测标签: {r['predicted_label']}  | 正确: {r['correct']}")
print(f"原始输出: '{r['raw_output']}'")

真实标签: positive  | 预测标签: positive  | 正确: True
原始输出: 'positive'


In [ ]:
#  更新评估循环支持Qwen
def run_evaluation(eval_dataset, dataset_name,
                   model_name="bart-large-mnli",
                   prompting_strategy="zero_shot",
                   temperature=0.0, seed=EVAL_SEED,
                   max_samples=None):

    dataset = eval_dataset
    if max_samples is not None:
        dataset = eval_dataset.select(range(max_samples))

    print(f"\n{'='*50}")
    print(f"模型: {model_name}  数据集: {dataset_name}  样本数: {len(dataset)}")
    print(f"{'='*50}")

    results = []
    for i, example in enumerate(dataset):
        if i % 10 == 0:
            print(f"  进度: {i}/{len(dataset)}...")

        if model_name == "bart-large-mnli":
            result = run_inference_bart(example, dataset_name)
        elif model_name == "qwen2.5-1.5b":
            result = run_inference_qwen(example, dataset_name)

        results.append(result)

    true_labels = [r["true_label"]      for r in results]
    pred_labels = [r["predicted_label"] for r in results]

    valid = [(t, p) for t, p in zip(true_labels, pred_labels)
             if p not in ("PARSE_ERROR", "API_ERROR")]
    true_v, pred_v = zip(*valid)

    acc = accuracy_score(true_v, pred_v)
    f1  = f1_score(true_v, pred_v, average="macro")

    print(f"\n  Accuracy : {acc:.4f} ({acc*100:.1f}%)")
    print(f"  F1-macro : {f1:.4f}")
    print(f"  解析错误 : {sum(1 for p in pred_labels if p == 'PARSE_ERROR')} 条")

    save_experiment_log(model_name, dataset_name, prompting_strategy,
                        temperature, seed, results, acc, f1)
    return acc, f1, results

In [ ]:
# 10条测试
acc, f1, results = run_evaluation(
    eval_dataset=sst2_eval,
    dataset_name="sst2",
    model_name="qwen2.5-1.5b",
    max_samples=10
)

print("\n前3条详细：")
for r in results[:3]:
    print(f"  真实:{r['true_label']:10s} | 预测:{r['predicted_label']:10s} | 正确:{r['correct']} | 原始输出:'{r['raw_output']}'")


模型: qwen2.5-1.5b  数据集: sst2  样本数: 10
  进度: 0/10...

  Accuracy : 0.9000 (90.0%)
  F1-macro : 0.8901
  解析错误 : 0 条
Log has been stored: experiment_logs/qwen2.5-1.5b_sst2_zero_shot_20260317_040616.json

前3条详细：
  真实:positive   | 预测:positive   | 正确:True | 原始输出:'positive'
  真实:positive   | 预测:positive   | 正确:True | 原始输出:'positive'
  真实:positive   | 预测:positive   | 正确:True | 原始输出:'positive'


In [ ]:
# 全量测试（100条）
# Qwen全量评估：3个数据集各100条
print("开始Qwen全量评估...")

acc_sst2, f1_sst2, _ = run_evaluation(
    sst2_eval, "sst2", model_name="qwen2.5-1.5b", max_samples=None
)

acc_ag, f1_ag, _ = run_evaluation(
    ag_eval, "ag_news", model_name="qwen2.5-1.5b", max_samples=None
)

acc_imdb, f1_imdb, _ = run_evaluation(
    imdb_eval, "imdb", model_name="qwen2.5-1.5b", max_samples=None
)

print("\n" + "="*50)
print("Qwen2.5-1.5B zero-shot 结果汇总")
print("="*50)
print(f"{'数据集':<12} {'Accuracy':>10} {'F1-macro':>10} {'Majority':>10}")
print(f"{'─'*42}")
print(f"{'SST-2':<12} {acc_sst2:>10.4f} {f1_sst2:>10.4f} {'0.5700':>10}")
print(f"{'AG News':<12} {acc_ag:>10.4f} {f1_ag:>10.4f} {'0.3200':>10}")
print(f"{'IMDb':<12} {acc_imdb:>10.4f} {f1_imdb:>10.4f} {'0.5300':>10}")

开始Qwen全量评估...

模型: qwen2.5-1.5b  数据集: sst2  样本数: 100
  进度: 0/100...
  进度: 10/100...
  进度: 20/100...
  进度: 30/100...
  进度: 40/100...
  进度: 50/100...
  进度: 60/100...
  进度: 70/100...
  进度: 80/100...
  进度: 90/100...

  Accuracy : 0.9100 (91.0%)
  F1-macro : 0.9093
  解析错误 : 0 条
Log has been stored: experiment_logs/qwen2.5-1.5b_sst2_zero_shot_20260317_040635.json

模型: qwen2.5-1.5b  数据集: ag_news  样本数: 100
  进度: 0/100...
  进度: 10/100...
  进度: 20/100...
  进度: 30/100...
  进度: 40/100...
  进度: 50/100...
  进度: 60/100...
  进度: 70/100...
  进度: 80/100...
  进度: 90/100...

  Accuracy : 0.7475 (74.7%)
  F1-macro : 0.7146
  解析错误 : 1 条
Log has been stored: experiment_logs/qwen2.5-1.5b_ag_news_zero_shot_20260317_040658.json

模型: qwen2.5-1.5b  数据集: imdb  样本数: 100
  进度: 0/100...
  进度: 10/100...
  进度: 20/100...
  进度: 30/100...
  进度: 40/100...
  进度: 50/100...
  进度: 60/100...
  进度: 70/100...
  进度: 80/100...
  进度: 90/100...

  Accuracy : 0.9400 (94.0%)
  F1-macro : 0.9400
  解析错误 : 0 条
Log has been stored: experim

In [ ]:
# ============================================================
# 构造Few-shot Exemplar集合
#
# 设计原则：
# 1. 每个类别各取1个exemplar，保证类别均衡
# 2. 从train set里取，不能用eval set里的数据
# 3. 用固定seed，保证所有实验复用同一套exemplar
# ============================================================

def build_exemplars(dataset_name, n_per_class=1, seed=EVAL_SEED):
    """
    从train set里每个类别抽n_per_class条作为exemplar。
    返回exemplar列表，每条包含formatted_prompt和true_label_name。
    """
    if dataset_name == "sst2":
        train_data = sst2["train"]
    elif dataset_name == "ag_news":
        train_data = ag["train"]
    elif dataset_name == "imdb":
        train_data = imdb["train"]

    # 预处理train set
    train_pp = train_data.map(lambda x: preprocess_data(x, dataset_name))

    # 获取所有类别
    all_labels = list(LABEL_MAPPINGS[dataset_name].keys())

    exemplars = []
    for label_id in all_labels:
        # 筛选出这个类别的所有样本
        label_samples = train_pp.filter(
            lambda x: int(x["true_label_id"]) == label_id
        )
        # 用固定seed抽取n_per_class条
        selected = label_samples.shuffle(seed=seed).select(range(n_per_class))
        for ex in selected:
            exemplars.append({
                "raw_text"       : ex["raw_text"],
                "true_label_name": ex["true_label_name"]
            })

    return exemplars

# 构造3个数据集的exemplar（每类1条，SST-2共2条，AG News共4条，IMDb共2条）
exemplars_sst2 = build_exemplars("sst2")
exemplars_ag   = build_exemplars("ag_news")
exemplars_imdb = build_exemplars("imdb")

print("Exemplar构造完成 ✓")
print(f"\nSST-2 exemplars ({len(exemplars_sst2)}条):")
for ex in exemplars_sst2:
    print(f"  [{ex['true_label_name']}] {ex['raw_text'][:60]}...")

print(f"\nAG News exemplars ({len(exemplars_ag)}条):")
for ex in exemplars_ag:
    print(f"  [{ex['true_label_name']}] {ex['raw_text'][:60]}...")

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Filter:   0%|          | 0/67349 [00:00<?, ? examples/s]

Filter:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/120000 [00:00<?, ? examples/s]

...

Filter:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/25000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/25000 [00:00<?, ? examples/s]

Exemplar构造完成 ✓

SST-2 exemplars (2条):
  [negative] its appeal will probably limited to lds church members and u...
  [positive] this familiar rise-and-fall tale is long on glamour and shor...

AG News exemplars (4条):
  [World] Rocket explodes at scrapyard, killing 10 GHAZIABAD: Thursday...
  [Sports] Verplank Will Play in World Cup (AP) AP - Scott Verplank had...
  [Business] Bundesbank sees German economy growing by 1.2 - 1.3 pct in 2...
  [Sci/Tech] Netflix, TiVo Make Deal Official Netflix Inc. and TiVo Inc. ...


In [ ]:
# ============================================================
# Few-shot Prompt构造函数
#
# 把exemplar列表拼接到prompt前面
# exemplars参数控制顺序，后面顺序扰动实验就是改这个参数
# ============================================================

def build_fewshot_prompt(example, dataset_name, exemplars):
    """
    构造few-shot prompt。
    exemplars: exemplar列表，顺序由调用方控制
    """
    prompt_parts = []

    # 每个exemplar都用同样的模板格式化，并加上答案
    for ex in exemplars:
        ex_prompt = PROMPT_TEMPLATES[dataset_name].format(input=ex["raw_text"])
        prompt_parts.append(f"{ex_prompt} {ex['true_label_name']}")

    # 最后加上待预测的样本（不加答案）
    query_prompt = PROMPT_TEMPLATES[dataset_name].format(
        input=example["raw_text"]
    )
    prompt_parts.append(query_prompt)

    # 用换行符连接所有部分
    return "\n\n".join(prompt_parts)

# 测试一下，看看few-shot prompt长什么样
test_prompt = build_fewshot_prompt(sst2_eval[0], "sst2", exemplars_sst2)
print("Few-shot prompt示例：")
print("─"*50)
print(test_prompt)
print("─"*50)

Few-shot prompt示例：
──────────────────────────────────────────────────
Classify the sentiment of the following review:
its appeal will probably limited to lds church members and undemanding armchair tourists .
Label: negative

Classify the sentiment of the following review:
this familiar rise-and-fall tale is long on glamour and short on larger moralistic consequences , though it 's told with sharp ears and eyes for the tenor of the times .
Label: positive

Classify the sentiment of the following review:
it gets onto the screen just about as much of the novella as one could reasonably expect , and is engrossing and moving in its own right .
Label:
──────────────────────────────────────────────────


In [ ]:
# 现在加few-shot推理函数和顺序扰动实验：
# ============================================================
# Few-shot推理函数（Qwen版本）
# 和zero-shot的区别只是prompt换成了few-shot格式
# ============================================================

def run_inference_qwen_fewshot(example, dataset_name, exemplars):
    """Few-shot版本，exemplars顺序由外部控制"""

    prompt = build_fewshot_prompt(example, dataset_name, exemplars)

    messages = [
        {
            "role": "system",
            "content": "You are a text classification assistant. Reply with only the label, nothing else."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors="pt").to(model_qwen.device)

    t_start = time.time()
    with torch.no_grad():
        outputs = model_qwen.generate(
            **inputs,
            max_new_tokens=10,
            temperature=0.01,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    latency = time.time() - t_start

    input_length = inputs["input_ids"].shape[1]
    generated_ids = outputs[0][input_length:]
    raw_output = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    predicted_label = parse_label(raw_output, dataset_name)

    return {
        "predicted_label" : predicted_label,
        "true_label"      : example["true_label_name"],
        "correct"         : predicted_label == example["true_label_name"],
        "latency_sec"     : round(latency, 4),
        "tokens_used"     : len(generated_ids),
        "raw_output"      : raw_output
    }

# 测试单条
r = run_inference_qwen_fewshot(sst2_eval[0], "sst2", exemplars_sst2)
print(f"真实标签: {r['true_label']}  | 预测标签: {r['predicted_label']}  | 正确: {r['correct']}")
print(f"原始输出: '{r['raw_output']}'")

真实标签: positive  | 预测标签: positive  | 正确: True
原始输出: 'positive'


In [ ]:
# ============================================================
# 顺序扰动实验
#
# 对SST-2的2个exemplar，有2种排列（2! = 2）
# 对AG News的4个exemplar，有24种排列（4! = 24），太多了取6种
# 每种排列跑100条，记录accuracy
# 最后看accuracy的标准差，标准差越大说明顺序影响越大
# ============================================================

from itertools import permutations
import numpy as np

def run_order_experiment(eval_dataset, dataset_name, exemplars,
                         max_orders=6, max_samples=50):
    """
    对exemplar的不同排列顺序各跑一次评估。
    max_orders: 最多测试几种排列（4!= 24种太多，取前6种）
    max_samples: 每种排列跑多少条（节省时间用50条）
    """
    all_orders = list(permutations(range(len(exemplars))))

    # 如果排列太多就只取前max_orders种
    if len(all_orders) > max_orders:
        # 用固定seed随机抽取，保证可复现
        rng = np.random.default_rng(EVAL_SEED)
        indices = rng.choice(len(all_orders), max_orders, replace=False)
        selected_orders = [all_orders[i] for i in indices]
    else:
        selected_orders = all_orders

    print(f"\n顺序扰动实验: {dataset_name}")
    print(f"测试排列数: {len(selected_orders)}  每排列样本数: {max_samples}")
    print("─"*40)

    accuracies = []
    dataset = eval_dataset.select(range(max_samples))

    for order_idx, order in enumerate(selected_orders):
        # 按当前排列重新组织exemplar
        ordered_exemplars = [exemplars[i] for i in order]
        order_str = str([exemplars[i]['true_label_name'] for i in order])

        results = []
        for example in dataset:
            r = run_inference_qwen_fewshot(example, dataset_name, ordered_exemplars)
            results.append(r)

        # 计算这种排列的accuracy
        valid = [(r["true_label"], r["predicted_label"])
                 for r in results if r["predicted_label"] != "PARSE_ERROR"]

        if valid:
            true_v, pred_v = zip(*valid)
            acc = accuracy_score(true_v, pred_v)
        else:
            acc = 0.0

        accuracies.append(acc)
        print(f"  排列 {order_idx+1}: {order_str}")
        print(f"           Accuracy: {acc:.4f}")

    # 计算统计指标
    mean_acc = np.mean(accuracies)
    std_acc  = np.std(accuracies)

    print(f"\n  均值: {mean_acc:.4f}")
    print(f"  标准差: {std_acc:.4f}  ← 越大说明顺序影响越大")
    print(f"  最高: {max(accuracies):.4f}  最低: {min(accuracies):.4f}")
    print(f"  波动范围: {max(accuracies)-min(accuracies):.4f}")

    return accuracies, mean_acc, std_acc

In [ ]:
accs_sst2, mean_sst2, std_sst2 = run_order_experiment(
    sst2_eval, "sst2", exemplars_sst2,
    max_orders=6,
    max_samples=50
)


顺序扰动实验: sst2
测试排列数: 2  每排列样本数: 50
────────────────────────────────────────
  排列 1: ['negative', 'positive']
           Accuracy: 0.9000
  排列 2: ['positive', 'negative']
           Accuracy: 0.9167

  均值: 0.9083
  标准差: 0.0083  ← 越大说明顺序影响越大
  最高: 0.9167  最低: 0.9000
  波动范围: 0.0167


In [ ]:
accs_ag, mean_ag, std_ag = run_order_experiment(
    ag_eval, "ag_news", exemplars_ag,
    max_orders=6,
    max_samples=50
)


顺序扰动实验: ag_news
测试排列数: 6  每排列样本数: 50
────────────────────────────────────────
  排列 1: ['Business', 'World', 'Sci/Tech', 'Sports']
           Accuracy: 0.7917
  排列 2: ['Business', 'Sports', 'Sci/Tech', 'World']
           Accuracy: 0.7755
  排列 3: ['Sci/Tech', 'Business', 'World', 'Sports']
           Accuracy: 0.7917
  排列 4: ['Sci/Tech', 'Sports', 'World', 'Business']
           Accuracy: 0.8085
  排列 5: ['Sports', 'Business', 'Sci/Tech', 'World']
           Accuracy: 0.8085
  排列 6: ['World', 'Sports', 'Sci/Tech', 'Business']
           Accuracy: 0.8085

  均值: 0.7974
  标准差: 0.0124  ← 越大说明顺序影响越大
  最高: 0.8085  最低: 0.7755
  波动范围: 0.0330


In [ ]:
accs_imdb, mean_imdb, std_imdb = run_order_experiment(
    imdb_eval, "imdb", exemplars_imdb,
    max_orders=6,
    max_samples=50
)


顺序扰动实验: imdb
测试排列数: 2  每排列样本数: 50
────────────────────────────────────────
  排列 1: ['negative', 'positive']
           Accuracy: 0.9787
  排列 2: ['positive', 'negative']
           Accuracy: 0.9787

  均值: 0.9787
  标准差: 0.0000  ← 越大说明顺序影响越大
  最高: 0.9787  最低: 0.9787
  波动范围: 0.0000


In [ ]:
print("="*55)
print("顺序扰动实验汇总")
print("="*55)
print(f"{'数据集':<12} {'分类数':>6} {'均值':>8} {'标准差':>8} {'波动范围':>10}")
print(f"{'─'*55}")
print(f"{'SST-2':<12} {'2':>6} {mean_sst2:>8.4f} {std_sst2:>8.4f} {max(accs_sst2)-min(accs_sst2):>10.4f}")
print(f"{'AG News':<12} {'4':>6} {mean_ag:>8.4f} {std_ag:>8.4f} {max(accs_ag)-min(accs_ag):>10.4f}")
print(f"{'IMDb':<12} {'2':>6} {mean_imdb:>8.4f} {std_imdb:>8.4f} {max(accs_imdb)-min(accs_imdb):>10.4f}")
print(f"{'─'*55}")
print("\n结论：分类数越多，顺序影响越大")

顺序扰动实验汇总
数据集             分类数       均值      标准差       波动范围
───────────────────────────────────────────────────────
SST-2             2   0.9083   0.0083     0.0167
AG News           4   0.7974   0.0124     0.0330
IMDb              2   0.9787   0.0000     0.0000
───────────────────────────────────────────────────────

结论：分类数越多，顺序影响越大


你可以得出这样的结论：在简单二分类任务上（尤其是长文本），exemplar顺序对Qwen2.5-1.5B的影响可以忽略不计。但在多分类任务上，顺序会带来最高3.3%的accuracy波动，在资源受限的小样本评估场景下这是不可忽视的。这直接呼应了你proposal里引用的Zhao et al. 2021，但你的实验给出了更细致的结论：影响大小取决于任务复杂度和文本长度。


In [ ]:
# Zero-shot vs Few-shot的对比
# ============================================================
# Few-shot全量评估（用固定顺序的exemplar）
# 和之前zero-shot的100条结果对比
# ============================================================

def run_fewshot_evaluation(eval_dataset, dataset_name, exemplars,
                           model_name="qwen2.5-1.5b",
                           max_samples=None):

    dataset = eval_dataset
    if max_samples is not None:
        dataset = eval_dataset.select(range(max_samples))

    print(f"\n{'='*50}")
    print(f"Few-shot | 模型: {model_name} | 数据集: {dataset_name} | 样本数: {len(dataset)}")
    print(f"{'='*50}")

    results = []
    for i, example in enumerate(dataset):
        if i % 10 == 0:
            print(f"  进度: {i}/{len(dataset)}...")
        r = run_inference_qwen_fewshot(example, dataset_name, exemplars)
        results.append(r)

    true_labels = [r["true_label"]      for r in results]
    pred_labels = [r["predicted_label"] for r in results]

    valid = [(t, p) for t, p in zip(true_labels, pred_labels)
             if p != "PARSE_ERROR"]
    true_v, pred_v = zip(*valid)

    acc = accuracy_score(true_v, pred_v)
    f1  = f1_score(true_v, pred_v, average="macro")

    print(f"\n  Accuracy : {acc:.4f} ({acc*100:.1f}%)")
    print(f"  F1-macro : {f1:.4f}")
    print(f"  解析错误 : {sum(1 for p in pred_labels if p == 'PARSE_ERROR')} 条")

    save_experiment_log(
        model_name=f"{model_name}-fewshot",
        dataset_name=dataset_name,
        prompting_strategy="few_shot",
        temperature=0.0,
        seed=EVAL_SEED,
        results=results,
        accuracy=acc,
        f1=f1
    )
    return acc, f1, results

# 跑三个数据集的few-shot评估
acc_fs_sst2, f1_fs_sst2, _ = run_fewshot_evaluation(
    sst2_eval, "sst2", exemplars_sst2
)
acc_fs_ag, f1_fs_ag, _ = run_fewshot_evaluation(
    ag_eval, "ag_news", exemplars_ag
)
acc_fs_imdb, f1_fs_imdb, _ = run_fewshot_evaluation(
    imdb_eval, "imdb", exemplars_imdb
)

# 最终对比表
print("\n" + "="*60)
print("Zero-shot vs Few-shot 完整对比（Qwen2.5-1.5B）")
print("="*60)
print(f"{'数据集':<12} {'Zero-shot':>12} {'Few-shot':>12} {'变化':>10}")
print(f"{'─'*50}")
print(f"{'SST-2':<12} {acc_sst2:>12.4f} {acc_fs_sst2:>12.4f} {acc_fs_sst2-acc_sst2:>+10.4f}")
print(f"{'AG News':<12} {acc_ag:>12.4f} {acc_fs_ag:>12.4f} {acc_fs_ag-acc_ag:>+10.4f}")
print(f"{'IMDb':<12} {acc_imdb:>12.4f} {acc_fs_imdb:>12.4f} {acc_fs_imdb-acc_imdb:>+10.4f}")


Few-shot | 模型: qwen2.5-1.5b | 数据集: sst2 | 样本数: 100
  进度: 0/100...
  进度: 10/100...
  进度: 20/100...
  进度: 30/100...
  进度: 40/100...
  进度: 50/100...
  进度: 60/100...
  进度: 70/100...
  进度: 80/100...
  进度: 90/100...

  Accuracy : 0.9100 (91.0%)
  F1-macro : 0.9096
  解析错误 : 0 条
Log has been stored: experiment_logs/qwen2.5-1.5b-fewshot_sst2_few_shot_20260317_040846.json

Few-shot | 模型: qwen2.5-1.5b | 数据集: ag_news | 样本数: 100
  进度: 0/100...
  进度: 10/100...
  进度: 20/100...
  进度: 30/100...
  进度: 40/100...
  进度: 50/100...
  进度: 60/100...
  进度: 70/100...
  进度: 80/100...
  进度: 90/100...

  Accuracy : 0.7938 (79.4%)
  F1-macro : 0.7512
  解析错误 : 3 条
Log has been stored: experiment_logs/qwen2.5-1.5b-fewshot_ag_news_few_shot_20260317_040901.json

Few-shot | 模型: qwen2.5-1.5b | 数据集: imdb | 样本数: 100
  进度: 0/100...
  进度: 10/100...
  进度: 20/100...
  进度: 30/100...
  进度: 40/100...
  进度: 50/100...
  进度: 60/100...
  进度: 70/100...
  进度: 80/100...
  进度: 90/100...

  Accuracy : 0.9468 (94.7%)
  F1-macro : 0.9458
  


IMDb的few-shot有6条解析错误，zero-shot是0条。这说明prompt越长，模型输出越不稳定，这本身就是一个reproducibility的发现，值得在论文里提到。

In [ ]:
import shutil

# 把experiment_logs文件夹打包成zip
shutil.make_archive("experiment_logs_backup", "zip", "experiment_logs")
print("打包完成 ✓")
print("请在左侧文件栏找到 experiment_logs_backup.zip 下载")

打包完成 ✓
请在左侧文件栏找到 experiment_logs_backup.zip 下载
